# Visualizing the Krogh-Cylinder Cross-Section

The PDE solved in this repo (`data/synthetic/solver.py`) is 1D, but the spatial variable `x` is not a straight line through the tissue — it is the **radial distance from the vessel wall**. The model assumes axisymmetric diffusion (no angular dependence, no variation along the vessel's length), so concentration only depends on that radial distance `x` and time `t`: `C(x,t)`.

This notebook turns the 1D profile `C(x,t)` into an actual 2D picture of the tissue cross-section: a disk with the vessel at its center (`x=0`) and the outer edge of the Krogh cylinder at radius `x=L` — the same schematic as Fig. 1A in Thurber et al. and Vasalou et al.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, so `data.synthetic.solver` is importable

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

from data.synthetic.solver import analytical_transient_profile, steady_state_profile


## 2. Pick physical parameters

Same defaults as the `solver.py` self-check: `D` (diffusion coefficient), `r` (decay rate), `c0` (concentration at the vessel wall), `L` (Krogh cylinder radius). Try changing `r` — a larger `r` means faster loss, so the drug stays confined closer to the vessel (smaller penetration depth `λ=√(D/r)`), which you'll see as a smaller bright region in the disk view below.


In [ ]:
D, r, c0, L = 10.0, 1e-3, 1.0, 100.0   # um^2/s, 1/s, concentration unit, um
nx = 200
x_grid = np.linspace(0.0, L, nx)

lam = np.sqrt(D / r)
print(f"Penetration depth lambda = sqrt(D/r) = {lam:.1f} um  (domain L = {L:.0f} um)")


## 3. Classic 1D profile — concentration vs. radial distance

In [ ]:
t_snapshots = [50.0, 500.0, 2000.0, 8000.0, 30000.0, 100000.0]

fig, ax = plt.subplots(figsize=(6, 4))
for t in t_snapshots:
    C = analytical_transient_profile(x_grid, t, D, r, c0, L, n_modes=200)
    ax.plot(x_grid, C, label=f"t={t:,.0f} s")
ax.plot(x_grid, steady_state_profile(x_grid, D, r, c0, L), "k--", label="steady state")
ax.set_xlabel("x = radial distance from vessel wall (um)")
ax.set_ylabel("C(x,t)")
ax.set_title("1D radial profile")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()


## 4. From radial profile to a 2D cross-section

Since concentration only depends on radius, we can build a 2D image by computing, for every pixel `(X,Y)` in a square canvas, its distance from the center `R = sqrt(X^2+Y^2)`, then looking up `C(R,t)` by interpolating the 1D profile. Pixels beyond `R=L` (outside the Krogh cylinder — where the next capillary's territory begins) are masked out.


In [ ]:
def radial_profile_to_disk(x_grid, C, L, resolution=300):
    """Map a 1D radial profile C(x) onto a 2D disk of the given resolution."""
    lim = L * 1.05
    xx, yy = np.meshgrid(
        np.linspace(-lim, lim, resolution), np.linspace(-lim, lim, resolution)
    )
    R = np.sqrt(xx**2 + yy**2)
    disk = np.interp(R, x_grid, C)
    disk[R > L] = np.nan
    return disk, lim


## 5. Single snapshot — disk view at one time point

In [ ]:
t = 2000.0
C = analytical_transient_profile(x_grid, t, D, r, c0, L, n_modes=200)
disk, lim = radial_profile_to_disk(x_grid, C, L)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(disk, extent=[-lim, lim, -lim, lim], origin="lower", cmap="viridis",
               vmin=0, vmax=c0)
ax.add_patch(plt.Circle((0, 0), L * 0.03, color="red", zorder=3))  # vessel (schematic, not to scale)
ax.set_title(f"Krogh cylinder cross-section, t={t:,.0f} s")
ax.set_xlabel("um")
ax.set_ylabel("um")
fig.colorbar(im, ax=ax, label="C(x,t)")
plt.show()


## 6. Grid of snapshots over time

Side-by-side disks at increasing `t`, so you can see the drug penetrate outward from the vessel (center, red dot) toward the edge of the cylinder.


In [ ]:
t_grid = [10.0, 100.0, 1000.0, 5000.0, 20000.0, 100000.0]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, t in zip(axes.ravel(), t_grid):
    C = analytical_transient_profile(x_grid, t, D, r, c0, L, n_modes=200)
    disk, lim = radial_profile_to_disk(x_grid, C, L)
    im = ax.imshow(disk, extent=[-lim, lim, -lim, lim], origin="lower", cmap="viridis",
                   vmin=0, vmax=c0)
    ax.add_patch(plt.Circle((0, 0), L * 0.03, color="red", zorder=3))
    ax.set_title(f"t={t:,.0f} s")
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, label="C(x,t)", shrink=0.8)
plt.show()


## 7. Animation — watch the drug penetrate over time

Renders as an interactive HTML5/JS animation inline (no `ffmpeg` required). Left: the disk cross-section. Right: the same profile as a 1D curve, for reference.


In [ ]:
t_anim = np.logspace(0, np.log10(150000.0), 60)

fig, (ax_disk, ax_line) = plt.subplots(1, 2, figsize=(11, 5))

C0 = analytical_transient_profile(x_grid, t_anim[0], D, r, c0, L, n_modes=200)
disk0, lim = radial_profile_to_disk(x_grid, C0, L)
im = ax_disk.imshow(disk0, extent=[-lim, lim, -lim, lim], origin="lower", cmap="viridis",
                    vmin=0, vmax=c0)
ax_disk.add_patch(plt.Circle((0, 0), L * 0.03, color="red", zorder=3))
ax_disk.set_xticks([]); ax_disk.set_yticks([])
title = ax_disk.set_title(f"t={t_anim[0]:,.0f} s")

(line,) = ax_line.plot(x_grid, C0)
ax_line.set_xlim(0, L)
ax_line.set_ylim(0, c0)
ax_line.set_xlabel("x = radial distance (um)")
ax_line.set_ylabel("C(x,t)")
ax_line.grid(alpha=0.3)
fig.tight_layout()


def update(frame):
    t = t_anim[frame]
    C = analytical_transient_profile(x_grid, t, D, r, c0, L, n_modes=200)
    disk, _ = radial_profile_to_disk(x_grid, C, L)
    im.set_data(disk)
    line.set_ydata(C)
    title.set_text(f"t={t:,.0f} s")
    return im, line, title


anim = animation.FuncAnimation(fig, update, frames=len(t_anim), interval=80, blit=False)
plt.close(fig)  # prevent a duplicate static plot below the animation
HTML(anim.to_jshtml())


## 8. 3D surface: x, t, and C(x,t) together

Instead of one curve per time snapshot, put `x` and `t` on the two horizontal axes and `C(x,t)` on the vertical axis — the entire dataset that `generate_data.py` samples from, visualized as one surface (the same style of plot as Fig. 2 in Vasalou et al. 2015).


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3D projection)

t_surface = np.logspace(0, np.log10(150000.0), 80)
C_surface = analytical_transient_profile(x_grid, t_surface, D, r, c0, L, n_modes=200)  # (nx, nt)

X, T = np.meshgrid(x_grid, t_surface, indexing="ij")

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, T, C_surface, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("x = radial distance (um)")
ax.set_ylabel("t (s)")
ax.set_zlabel("C(x,t)")
ax.set_title("Concentration surface over space and time")
ax.view_init(elev=25, azim=-135)  # rotate/re-run to view from a different angle
fig.colorbar(surf, ax=ax, shrink=0.6, label="C(x,t)")
plt.show()
